# 01b — HRV integrity and duplication review

**Inputs:** `outputs/dataset_clean.csv` and `outputs/data_errors.csv`, produced by Notebook 01.  
**Exactly two outputs:** `outputs/dataset_review_clear.csv` and `outputs/dataset_review_suspected.csv`.

## Purpose and interpretation

Audit the entire cleaned source before categorization. **Every source row is assigned to exactly one output**. A record enters the suspected file whenever it has a record-level audit finding. Suspected records remain intact: no measurements are altered, no value is imputed, and no record is silently discarded.

- **Exact HRV profile:** all seven HRV measurements are present and identical across records.
- **HF/VLF exchange:** RR, HR, SDNN, RMSSD and LF agree exactly; HF and VLF are exchanged between records, with a nontrivial exchange.
- **Near HRV profile:** RR, HR, SDNN and RMSSD agree exactly and at least five of the seven HRV measurements agree; this is only an audit signal.
- **Partial HRV profile:** at least five comparable measures match exactly, all *available in both records* match, and at least one HRV measurement is missing; a missing entry does **not** count as equal.
- **Individual quality flags:** missing numerical measurements, implausible/invalid values, HR/RR disagreement, range-screening warnings, and source-level height/BMI corrections requiring verification. Numeric text successfully converted by Notebook 01 is *not* considered suspicious solely for its storage type.

Each suspected row keeps all the cleaned columns and receives a review group ID, related Excel row IDs, all reason codes, affected variables, and human-readable evidence. The same group ID is assigned to all members of a linked cluster; isolated observations receive a singleton group ID. Groups are constructed from discovered links, never from a hard-coded list of row pairs.

**Important:** 'clear' means no *individual, rule-detectable* anomaly was found. It does not certify clinical validity. Possible cohort-level scale/batch shifts (notably SDNN) are reported separately and are **not** converted into hundreds of unsupported individual error labels. This notebook is a review/triage stage, **not an eligibility filter**, and does not change the inputs used by the existing Notebooks 02–04 without an explicit decision.

## 1. Load and validate the cleaned source and audit log

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

INPUT_FILE = Path("../../outputs/dataset_clean.csv")
ERRORS_FILE = Path("../../outputs/data_errors.csv")
OUTPUT_DIR = Path("../../outputs")
CLEAR_FILE = OUTPUT_DIR / "dataset_review_clear.csv"
SUSPECTED_FILE = OUTPUT_DIR / "dataset_review_suspected.csv"

HRV = [
    "t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn",
    "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf",
]
CORE_HRV = HRV[:4]
NUMERIC = [
    "sex", "age", *HRV, "bp_systolic", "bp_diastolic",
    "bp_pam", "bp_pp", "imc", "weight_kg", "height_cm", "height_m",
]
CONTEXT = ["sex", "age", "bp_systolic", "bp_diastolic", "weight_kg", "height_cm", "imc"]
REVIEW_RANGES = {
    "age": (18, 100), "weight_kg": (30, 250), "height_cm": (120, 230),
    "height_m": (1.2, 2.3), "imc": (10, 80),
    "bp_systolic": (70, 250), "bp_diastolic": (40, 150),
    "t2m_pre_mean_hr": (30, 220), "t2m_pre_mean_rr": (250, 2000),
    "t2m_pre_sdnn": (0, 500), "t2m_pre_rmssd": (0, 500),
    "t2m_pre_hf": (0, 100000), "t2m_pre_lf": (0, 100000),
    "t2m_pre_vlf": (0, 100000),
}
HR_RR_REVIEW_THRESHOLD_BPM = 2.0
HR_RR_FACTOR_TEN_RTOL = 0.03
MIN_SHARED_HRV_FOR_PAIR = 5

if not INPUT_FILE.is_file() or not ERRORS_FILE.is_file():
    raise FileNotFoundError("Run Notebook 01 first: dataset_clean.csv and data_errors.csv are required.")
source = pd.read_csv(INPUT_FILE)
error_log = pd.read_csv(ERRORS_FILE)
required = ["source_excel_row", *NUMERIC]
missing_columns = sorted(set(required) - set(source.columns))
if missing_columns:
    raise ValueError(f"Missing required fields: {missing_columns}")
if source["source_excel_row"].isna().any() or not source["source_excel_row"].is_unique:
    raise ValueError("source_excel_row must contain unique, non-null Excel row numbers.")
if not pd.api.types.is_integer_dtype(source["source_excel_row"]):
    raise ValueError("source_excel_row must contain integer Excel row numbers.")
for col in NUMERIC:
    if not pd.api.types.is_numeric_dtype(source[col]):
        raise TypeError(f"Expected a numeric cleaned column: {col}")
for col in ["source_excel_row", "column", "issue", "original_value", "processed_value", "action"]:
    if col not in error_log:
        raise ValueError(f"Missing expected audit-log column: {col}")

source_columns = source.columns.tolist()
by_excel_row = source.set_index("source_excel_row", drop=False)
excel_rows = source["source_excel_row"].astype(int).tolist()
print(f"Loaded {len(source)} cleaned source records and {len(error_log)} Notebook 01 log entries.")
print("The original dataset and Notebook 01 outputs will not be overwritten.")

Loaded 530 cleaned source records and 183 Notebook 01 log entries.
The original dataset and Notebook 01 outputs will not be overwritten.


## 2. Identify individual inconsistencies without modifying values

In [2]:
# One record can accumulate several independent reasons.
findings = defaultdict(list)

def printable(value):
    if pd.isna(value):
        return "MISSING"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        return f"{float(value):.12g}"
    return str(value)

def add_finding(excel_row, code, fields, explanation):
    findings[int(excel_row)].append({
        "code": str(code),
        "fields": tuple(sorted(set(fields))),
        "explanation": str(explanation),
    })

for _, row in source.iterrows():
    excel_row = int(row["source_excel_row"])
    for col in NUMERIC:
        if pd.isna(row[col]):
            add_finding(excel_row, "MISSING_NUMERIC_MEASUREMENT", [col],
                        f"{col} is missing in the cleaned source")

    if pd.notna(row["sex"]) and row["sex"] not in (1, 2):
        add_finding(excel_row, "INVALID_SEX_CODE", ["sex"],
                    f"sex={printable(row['sex'])}; documented codes are 1=Female and 2=Male")
    if pd.notna(row["age"]) and not (0 <= row["age"] <= 120):
        add_finding(excel_row, "INVALID_AGE", ["age"], f"age={printable(row['age'])}")

    sbp, dbp = row["bp_systolic"], row["bp_diastolic"]
    if pd.notna(sbp) and pd.notna(dbp) and (sbp <= 0 or dbp <= 0 or sbp <= dbp):
        add_finding(excel_row, "INVALID_BP_RELATION", ["bp_systolic", "bp_diastolic"],
                    f"SBP={printable(sbp)}, DBP={printable(dbp)}; requires SBP>DBP>0")
    for col in ["imc", "weight_kg", "height_cm", "height_m", "t2m_pre_mean_rr", "t2m_pre_mean_hr"]:
        if pd.notna(row[col]) and row[col] <= 0:
            add_finding(excel_row, "NONPOSITIVE_MEASUREMENT", [col],
                        f"{col}={printable(row[col])}")
    for col in HRV[2:]:
        if pd.notna(row[col]) and row[col] < 0:
            add_finding(excel_row, "NEGATIVE_HRV", [col],
                        f"{col}={printable(row[col])}")

    for col, (low, high) in REVIEW_RANGES.items():
        value = row[col]
        if pd.notna(value) and (value < low or value > high):
            add_finding(excel_row, "OUTSIDE_REVIEW_RANGE", [col],
                        f"{col}={printable(value)} outside [{low}, {high}] (review warning, not proven error)")

    hr, rr = row["t2m_pre_mean_hr"], row["t2m_pre_mean_rr"]
    if pd.notna(hr) and pd.notna(rr) and hr > 0 and rr > 0:
        expected = 60000.0 / rr
        diff = abs(hr - expected)
        ratio = hr * rr / 60000.0
        tenfold = (np.isclose(ratio, 0.1, rtol=HR_RR_FACTOR_TEN_RTOL, atol=0) or
                   np.isclose(ratio, 10.0, rtol=HR_RR_FACTOR_TEN_RTOL, atol=0))
        if tenfold:
            add_finding(excel_row, "HR_RR_FACTOR_TEN_PATTERN", HRV[:2],
                        f"HR={printable(hr)} bpm; RR={printable(rr)} ms; HR*(RR)/60000={ratio:.5g}; "
                        f"expected HR from RR={expected:.3f} bpm")
        elif diff > HR_RR_REVIEW_THRESHOLD_BPM:
            add_finding(excel_row, "HR_RR_REVIEW_DISAGREEMENT", HRV[:2],
                        f"HR={printable(hr)} bpm; RR={printable(rr)} ms; "
                        f"expected HR from RR={expected:.3f} bpm; |difference|={diff:.3f} bpm")

# Retain information about source discrepancies already auto-corrected in Notebook 01.
# Successfully normalised numeric text and known duplicate notices are not added as new errors.
source_issue_codes = {
    "height_disagreement": "SOURCE_HEIGHT_DISAGREEMENT_CORRECTED",
    "bmi_disagreement": "SOURCE_BMI_DISAGREEMENT_CORRECTED",
    "unparseable_number": "UNPARSEABLE_SOURCE_NUMBER",
}
for _, log in error_log.iterrows():
    if pd.isna(log["source_excel_row"]):
        continue
    excel_row = int(log["source_excel_row"])
    if excel_row not in by_excel_row.index:
        raise ValueError(f"Notebook 01 audit log references unknown source row {excel_row}")
    code = source_issue_codes.get(log["issue"])
    if code is None:
        continue
    col = str(log["column"])
    detail = (f"Notebook 01 {log['issue']} in {col}: original={printable(log['original_value'])}, "
              f"processed={printable(log['processed_value'])}, action={printable(log['action'])}")
    add_finding(excel_row, code, [col], detail)

print(f"Individual-only suspicious records before pair matching: {len(findings)}")

Individual-only suspicious records before pair matching: 43


## 3. Discover exact, swapped, near, and partially matching HRV profiles

A link is generated from the measurements themselves. No previously identified row-pair list is used. **An exact match or apparent swap is evidence of unusual similarity, not proof of shared participant identity.**

In [3]:
# With 530 records, vectorized pair comparisons (~140,000 candidates) are modest.
measurements = source[HRV].to_numpy(dtype=float)
pair_links = []

def values_differ(x, y):
    if pd.isna(x) and pd.isna(y):
        return False
    if pd.isna(x) or pd.isna(y):
        return True
    return x != y

for i in range(len(source) - 1):
    left = measurements[i]
    right_mat = measurements[i+1:]
    comparable = np.isfinite(left)[None, :] & np.isfinite(right_mat)
    matches = comparable & (left[None, :] == right_mat)
    counts = matches.sum(axis=1)
    for offset in np.flatnonzero(counts >= MIN_SHARED_HRV_FOR_PAIR):
        j = i + int(offset) + 1
        right = measurements[j]
        both = comparable[offset]
        eq = matches[offset]
        shared = int(both.sum())
        matching = int(eq.sum())
        fully_observed = bool(both.all())
        rr_hr_sdnn_rmssd_equal = bool(eq[:4].all())
        full_equal = fully_observed and matching == 7
        hf_vlf_swapped = (
            fully_observed and rr_hr_sdnn_rmssd_equal and bool(eq[5])
            and left[4] == right[6] and left[6] == right[4]
            and left[4] != left[6]
        )
        if full_equal:
            code = "EXACT_SEVEN_HRV"
        elif hf_vlf_swapped:
            code = "HF_VLF_SWAPPED"
        elif matching == shared and shared < 7 and (not fully_observed):
            code = "PARTIAL_HRV_MATCH"
        elif rr_hr_sdnn_rmssd_equal and matching >= 5:
            code = "NEAR_HRV_MATCH"
        else:
            continue

        a, b = int(excel_rows[i]), int(excel_rows[j])
        same_fields = [HRV[k] for k in range(7) if eq[k]]
        diff_fields = [HRV[k] for k in range(7) if both[k] and not eq[k]]
        missing_fields = [HRV[k] for k in range(7) if not both[k]]
        original_a = source.iloc[i]
        original_b = source.iloc[j]
        diff_expl = "; ".join(
            f"{v}: row {a}={printable(original_a[v])} vs row {b}={printable(original_b[v])}"
            for v in diff_fields + missing_fields
        ) or "none"
        context_expl = "; ".join(
            f"{v}: row {a}={printable(original_a[v])} vs row {b}={printable(original_b[v])}"
            for v in CONTEXT if values_differ(original_a[v], original_b[v])
        ) or "none"
        evidence = (
            f"Rows {a}/{b}: {matching}/{shared} comparable HRV measures match; "
            f"matching=[{', '.join(same_fields)}]; "
            f"different or missing=[{diff_expl}]; "
            f"different context=[{context_expl}]"
        )
        if code == "HF_VLF_SWAPPED":
            evidence += (f"; verified HF/VLF exchange: "
                         f"{printable(original_a['t2m_pre_hf'])}/{printable(original_a['t2m_pre_vlf'])} "
                         f"versus {printable(original_b['t2m_pre_hf'])}/{printable(original_b['t2m_pre_vlf'])}")
        link = {
            "row_a": a, "row_b": b, "pattern": code, "matching_hrv_count": matching,
            "comparable_hrv_count": shared, "matching_hrv": ", ".join(same_fields),
            "different_hrv": ", ".join(diff_fields),
            "missing_hrv": ", ".join(missing_fields), "evidence": evidence,
        }
        pair_links.append(link)
        add_finding(a, code, HRV, evidence)
        add_finding(b, code, HRV, evidence)

pair_summary = Counter(item["pattern"] for item in pair_links)
print("Detected pair relationships:")
for code, count in sorted(pair_summary.items()):
    print(f"  {code}: {count}")
print(f"  TOTAL_PAIRS: {len(pair_links)}")

Detected pair relationships:
  EXACT_SEVEN_HRV: 8
  HF_VLF_SWAPPED: 33
  NEAR_HRV_MATCH: 3
  PARTIAL_HRV_MATCH: 1
  TOTAL_PAIRS: 45


## 4. Assign pair/group identifiers, including multi-link clusters

All records in a connected set of suspicious pairwise links receive the same `review_group_id`. A row without links but with an individual finding receives an independent singleton group ID. Every member of a pair remains in the suspected dataset.

In [4]:
# Disjoint-set/union-find keeps groups correct if future data yield three or more linked rows.
parent = {r: r for r in excel_rows}

def root(r):
    while parent[r] != r:
        parent[r] = parent[parent[r]]
        r = parent[r]
    return r

def unite(a, b):
    ra, rb = root(a), root(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

for link in pair_links:
    unite(link["row_a"], link["row_b"])

components = defaultdict(list)
for r in excel_rows:
    components[root(r)].append(r)
linked_clusters = sorted(
    (sorted(members) for members in components.values() if len(members) > 1),
    key=lambda members: members[0],
)
group_info = {}
for group_num, members in enumerate(linked_clusters, 1):
    gid = f"P{group_num:04d}"
    cluster_links = [link for link in pair_links if link["row_a"] in members and link["row_b"] in members]
    patterns = sorted({link["pattern"] for link in cluster_links})
    evidence = " || ".join(link["evidence"] for link in cluster_links)
    for r in members:
        group_info[r] = {
            "review_group_id": gid, "group_size": len(members),
            "group_members_excel_rows": ", ".join(map(str, members)),
            "group_pair_patterns": "; ".join(patterns),
            "group_pair_evidence": evidence,
        }

for r in sorted(findings):
    if r not in group_info:
        group_info[r] = {
            "review_group_id": f"S{r:04d}", "group_size": 1,
            "group_members_excel_rows": str(r), "group_pair_patterns": "",
            "group_pair_evidence": "",
        }

print(f"Pair/cluster groups: {len(linked_clusters)}")
print(f"Singleton review groups: {sum(g['group_size'] == 1 for g in group_info.values())}")
print(f"Unique records requiring individual review: {len(findings)}")

Pair/cluster groups: 45
Singleton review groups: 33
Unique records requiring individual review: 123


## 5. Generate two mutually exclusive datasets and verify all source rows are accounted for

In [5]:
flagged_rows = set(findings)
clear = source.loc[~source["source_excel_row"].isin(flagged_rows), source_columns].copy()
suspected = source.loc[source["source_excel_row"].isin(flagged_rows), source_columns].copy()

AUDIT_COLUMNS = [
    "review_status", "review_group_id", "group_size", "group_members_excel_rows",
    "group_pair_patterns", "review_reason_count", "review_reason_codes",
    "review_affected_columns", "review_justification", "group_pair_evidence",
]
metadata = []
for r in suspected["source_excel_row"].astype(int):
    issues = findings[r]
    codes = sorted(set(item["code"] for item in issues))
    affected = sorted({field for item in issues for field in item["fields"]})
    distinct_explanations = list(dict.fromkeys(item["explanation"] for item in issues))
    metadata.append({
        "review_status": "PENDING_SOURCE_REVIEW",
        **group_info[r],
        "review_reason_count": len(codes),
        "review_reason_codes": "; ".join(codes),
        "review_affected_columns": "; ".join(affected),
        "review_justification": " || ".join(distinct_explanations),
    })
meta = pd.DataFrame(metadata, index=suspected.index)
for field in AUDIT_COLUMNS:
    suspected[field] = meta[field]
suspected = suspected[AUDIT_COLUMNS + source_columns].sort_values(
    ["review_group_id", "source_excel_row"], kind="stable"
)
clear = clear.sort_values("source_excel_row", kind="stable")

# Strong partition and data-preservation assertions.
clear_ids = set(clear["source_excel_row"].astype(int))
suspected_ids = set(suspected["source_excel_row"].astype(int))
original_ids = set(excel_rows)
assert clear_ids.isdisjoint(suspected_ids)
assert clear_ids | suspected_ids == original_ids
assert len(clear) + len(suspected) == len(source)
assert clear.columns.tolist() == source_columns
assert set(source_columns).issubset(suspected.columns)
reassembled = pd.concat([clear[source_columns], suspected[source_columns]], ignore_index=True)
reassembled = reassembled.sort_values("source_excel_row").reset_index(drop=True)
expected_source = source.sort_values("source_excel_row").reset_index(drop=True)
pd.testing.assert_frame_equal(reassembled, expected_source, check_dtype=True)
assert suspected["review_reason_count"].ge(1).all()
assert suspected.groupby("review_group_id")["group_size"].first().eq(
    suspected.groupby("review_group_id").size()
).all()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clear.to_csv(CLEAR_FILE, index=False)
suspected.to_csv(SUSPECTED_FILE, index=False)

# Verify the actual saved files, not only in-memory tables.
clear_check = pd.read_csv(CLEAR_FILE)
suspected_check = pd.read_csv(SUSPECTED_FILE)
assert len(clear_check) == len(clear)
assert len(suspected_check) == len(suspected)
assert set(clear_check["source_excel_row"]).isdisjoint(set(suspected_check["source_excel_row"]))
assert set(clear_check["source_excel_row"]) | set(suspected_check["source_excel_row"]) == original_ids
print(f"CLEAR: {len(clear)} records, {len(clear.columns)} columns -> {CLEAR_FILE}")
print(f"SUSPECTED: {len(suspected)} records, {len(suspected.columns)} columns -> {SUSPECTED_FILE}")
print(f"PARTITION CHECK: {len(clear)} + {len(suspected)} = {len(source)}; no records lost or duplicated")

CLEAR: 407 records, 19 columns -> ../../outputs/dataset_review_clear.csv
SUSPECTED: 123 records, 29 columns -> ../../outputs/dataset_review_suspected.csv
PARTITION CHECK: 407 + 123 = 530; no records lost or duplicated


## 6. Review detected groups and individually flagged records

In [6]:
reason_summary = Counter(code for details in findings.values() for code in set(d["code"] for d in details))
print("Issue frequencies (not mutually exclusive):")
display(pd.DataFrame(
    sorted(reason_summary.items(), key=lambda item: (-item[1], item[0])),
    columns=["reason_code", "affected_records"],
))
print("Group overview (a group can contain more than two source rows):")
if not suspected.empty:
    display(suspected[["review_group_id", "group_size", "group_members_excel_rows", "group_pair_patterns"]]
            .drop_duplicates().head(20).reset_index(drop=True))
    highlighted = [22, 55, 39, 86, 114, 416, 422, 455, 514, 531]
    columns = ["review_group_id", "source_excel_row", "review_reason_codes", "group_members_excel_rows",
               "t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn", "t2m_pre_rmssd",
               "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf"]
    display(suspected.loc[suspected["source_excel_row"].isin(highlighted), columns]
            .sort_values("source_excel_row"))

Issue frequencies (not mutually exclusive):


,reason_code,affected_records
0,HF_VLF_SWAPPED,66
1,EXACT_SEVEN_HRV,16
2,MISSING_NUMERIC_MEASUREMENT,16
3,OUTSIDE_REVIEW_RANGE,14
4,HR_RR_FACTOR_TEN_PATTERN,11
5,SOURCE_BMI_DISAGREEMENT_CORRECTED,9
6,SOURCE_HEIGHT_DISAGREEMENT_CORRECTED,9
7,NEAR_HRV_MATCH,6
8,HR_RR_REVIEW_DISAGREEMENT,4
9,PARTIAL_HRV_MATCH,2


Group overview (a group can contain more than two source rows):


,review_group_id,group_size,group_members_excel_rows,group_pair_patterns
0,P0001,2,"22, 55",NEAR_HRV_MATCH
1,P0002,2,"37, 85",EXACT_SEVEN_HRV
2,P0003,2,"39, 86",NEAR_HRV_MATCH
3,P0004,2,"50, 92",EXACT_SEVEN_HRV
4,P0005,2,"53, 94",EXACT_SEVEN_HRV
5,P0006,2,"114, 416",NEAR_HRV_MATCH
6,P0007,2,"422, 455",HF_VLF_SWAPPED
7,P0008,2,"423, 456",HF_VLF_SWAPPED
8,P0009,2,"424, 457",HF_VLF_SWAPPED
9,P0010,2,"425, 458",HF_VLF_SWAPPED


,review_group_id,source_excel_row,review_reason_codes,group_members_excel_rows,t2m_pre_mean_rr,t2m_pre_mean_hr,t2m_pre_sdnn,t2m_pre_rmssd,t2m_pre_hf,t2m_pre_lf,t2m_pre_vlf
20,P0001,22,NEAR_HRV_MATCH,"22, 55",862.0,70.0,208.0,24.5,136,179,77
37,P0003,39,NEAR_HRV_MATCH,"39, 86",883.0,68.0,177.0,21.9,196,76,40
53,P0001,55,NEAR_HRV_MATCH,"22, 55",862.0,70.0,208.0,24.5,136,178,78
84,P0003,86,NEAR_HRV_MATCH,"39, 86",883.0,68.0,177.0,21.9,196,76,4
112,P0006,114,NEAR_HRV_MATCH,"114, 416",937.0,64.0,247.0,24.8,97,493,30
414,P0006,416,NEAR_HRV_MATCH,"114, 416",937.0,64.0,247.0,24.8,97,494,31
420,P0007,422,HF_VLF_SWAPPED,"422, 455",950.0,63.0,9.4,11.7,4,11,62
453,P0007,455,HF_VLF_SWAPPED,"422, 455",950.0,63.0,9.4,11.7,62,11,4
512,P0045,514,PARTIAL_HRV_MATCH,"514, 531",1016.0,59.0,18.9,16.9,604,189,16
529,P0045,531,MISSING_NUMERIC_MEASUREMENT; PARTIAL_HRV_MATCH,"514, 531",1016.0,NaN,18.9,16.9,604,189,16


## 7. Cohort-level SDNN scale diagnostic (report-only)

A sharp shift in the distribution of `SDNN/RMSSD` may indicate different measurement scales or acquisition batches. A population-level difference does not identify which individual values are erroneous. The diagnostic below **does not add record-level suspicion codes**, change measurements, or move observations between the two files. It prints the most pronounced split in row order, subject to a minimum segment size. Confirm any suspected scale issue against the source device/export protocol before deciding whether to amend the data.

In [7]:
valid = source[["source_excel_row", "t2m_pre_sdnn", "t2m_pre_rmssd"]].copy()
valid["ratio"] = valid["t2m_pre_sdnn"] / valid["t2m_pre_rmssd"]
valid.loc[~np.isfinite(valid["ratio"]) | (valid["ratio"] <= 0), "ratio"] = np.nan

ordered = valid.sort_values("source_excel_row").reset_index(drop=True)
min_segment = max(30, int(np.ceil(len(ordered) * 0.075)))
scored = []
for k in range(min_segment, len(ordered) - min_segment + 1):
    left = ordered["ratio"].iloc[:k].dropna()
    right = ordered["ratio"].iloc[k:].dropna()
    if len(left) < min_segment * 0.75 or len(right) < min_segment * 0.75:
        continue
    median_left, median_right = left.median(), right.median()
    if median_left > 0 and median_right > 0:
        scored.append((abs(np.log10(median_left / median_right)), k, median_left, median_right))
if scored:
    score, split, left_median, right_median = max(scored)
    split_a = int(ordered.iloc[split - 1]["source_excel_row"])
    split_b = int(ordered.iloc[split]["source_excel_row"])
    print(f"Largest row-order SDNN/RMSSD median contrast: Excel rows {split_a} / {split_b}")
    print(f"Median before: {left_median:.4g}; median after: {right_median:.4g}; "
          f"fold-change: {max(left_median/right_median, right_median/left_median):.2f}x")
    if score > np.log10(5):
        print("COHORT-LEVEL WARNING: >5x contrast. Examine raw SDNN scale/units and source batches.")
        print("This is not evidence that every record in either segment is individually incorrect.")
else:
    print("Insufficient comparable SDNN/RMSSD values to evaluate a row-order distribution change.")

print("Review complete. No automated correction or eligibility exclusion was performed.")

Largest row-order SDNN/RMSSD median contrast: Excel rows 421 / 422
Median before: 11.56; median after: 1.121; fold-change: 10.31x
COHORT-LEVEL WARNING: >5x contrast. Examine raw SDNN scale/units and source batches.
This is not evidence that every record in either segment is individually incorrect.
Review complete. No automated correction or eligibility exclusion was performed.
